In [1]:
import numpy as np
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
from gensim.models import Word2Vec

In [3]:
import torch
import math

# **self-attention class**

In [4]:
class SelfAttention:

    # to initialize it
    def __init__(self, dimen=4, random_w=True):

        self.dimen = dimen
        self.random_w = random_w

        if random_w:
            self.w_query = torch.rand(dimen, dimen)
            self.w_key = torch.rand(dimen, dimen)
            self.w_value = torch.rand(dimen, dimen)
        else:
            self.w_query = None
            self.w_key = None
            self.w_value = None

    def getQKV(self, static_embeddings):

        if self.random_w == False:
            
            queries = static_embeddings
            keys = static_embeddings
            values = static_embeddings
            
        else:
        
            queries = torch.matmul(static_embeddings, self.w_query)
            keys = torch.matmul(static_embeddings, self.w_key)
            values = torch.matmul(static_embeddings, self.w_value)

        return queries, keys, values

    def dotproduct(self, queries, keys):

        dot_product = torch.matmul(queries,keys.T)
        return dot_product

    def get_sqrt(self, dot_product):

        sqroot = math.sqrt(self.dimen)
        return (1/sqroot)*dot_product

    def apply_softmax(self, scaled_dp):
        
        weights = torch.softmax(scaled_dp.T, dim=0).T
        return weights

    def mul_weights_and_values(self, weights, values):

        contextual_embeddings = torch.matmul(weights, values)

        #print(contextual_embeddings)
        return contextual_embeddings
        

    # get embeddings of different sentences
    def __call__(self, static_embeddings):

        # 1. obtain query, key, value vectors
        queries, keys, values = self.getQKV(static_embeddings)

        # 2. perform dot product bw query and key
        dot_product = self.dotproduct(queries,keys)

        # 3. scale by 1/sqrt(dimen)
        scaled_dp = self.get_sqrt(dot_product)

        # 4. apply softmax to each row
        weights = self.apply_softmax(scaled_dp)

        # 5. multiply w value to get final embeddings
        contextual_embeddings = self.mul_weights_and_values(weights,values)
        return contextual_embeddings

In [5]:
sentence1 = "River bank flows"
s1 = sentence1.split(" ")
print(s1)

['River', 'bank', 'flows']


In [6]:
sentence2 = "Money bank grows"
s2 = sentence2.split(" ")
print(s2)

['Money', 'bank', 'grows']


In [7]:
sentences = [s1, s2]
print(sentences)

[['River', 'bank', 'flows'], ['Money', 'bank', 'grows']]


In [8]:
model = Word2Vec(sentences, vector_size=256, window=5, min_count=1, sg=1)

In [9]:
input_matrix1 = []

for word in s1:
    embedding = model.wv[word]
    input_matrix1.append(embedding)

np_ip1 = np.array(input_matrix1)
static_embeddings1 = torch.tensor(np_ip1)

#print(query_tensors)

In [10]:
sa_block = SelfAttention(256,True)

In [11]:
ce1 = sa_block(static_embeddings1)

In [12]:
input_matrix2 = []

for word in s2:
    embedding = model.wv[word]
    input_matrix2.append(embedding)

np_ip2 = np.array(input_matrix2)
static_embeddings2 = torch.tensor(np_ip2)

In [13]:
ce2 = sa_block(static_embeddings2)

In [14]:
print(static_embeddings2[0:2])

tensor([[-2.8321e-03, -3.7513e-03, -1.0717e-03, -3.2667e-03, -2.3589e-03,
         -2.2152e-03, -9.1568e-04, -6.6680e-04, -3.4988e-03, -2.8719e-04,
          3.1846e-03,  3.0041e-03, -2.8149e-03, -1.4324e-03,  1.2182e-03,
         -3.7386e-03,  5.7673e-04,  2.5486e-03,  2.2447e-03, -3.4231e-03,
         -1.7645e-03, -3.1798e-03,  1.7952e-05,  3.6186e-03,  2.3333e-03,
          1.9794e-03,  1.9770e-03, -1.2668e-03,  3.7313e-03, -2.8736e-03,
         -2.8400e-03, -8.8492e-04, -3.0413e-04, -1.2563e-03, -2.3148e-04,
          2.9253e-03, -2.7247e-04, -6.3474e-04,  1.0720e-03, -3.2653e-03,
          3.0687e-03,  3.3344e-03, -3.7438e-03,  9.5557e-04,  3.8691e-03,
         -2.9945e-03, -2.7215e-03, -3.0221e-03,  3.2797e-03, -2.6615e-04,
          3.5720e-03, -3.1868e-03,  1.4621e-03,  1.0293e-03,  2.9012e-04,
          9.0925e-04, -2.9176e-03, -3.6556e-03,  9.1976e-04,  2.4017e-03,
          3.1194e-03,  2.2406e-03, -3.0365e-04,  3.2446e-03, -3.6470e-03,
          1.3305e-03,  1.0420e-04,  1.